# Bias Visualization

In [1]:
import os
from pathlib import Path
from typing import Any

import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
from plotly.subplots import make_subplots

In [2]:
# MIA and DCR vs. Train Steps
train_steps_dcr = [0.66060, 0.67230, 0.68049, 0.67850, 0.70130, 0.71829, 0.74869, 0.76339, 0.77380, 0.77319]
train_steps_wb = [0.09900, 0.12400, 0.13600, 0.17300, 0.26000, 0.37300, 0.42500, 0.48000, 0.49000, 0.51500]
train_steps_bb = [0.10900, 0.11800, 0.11100, 0.10900, 0.11300, 0.15300, 0.20000, 0.24700, 0.28400, 0.27800]
train_steps = [5e03, 1e04, 1.5e04, 2.5e04, 5e04, 1e05, 2e05, 3e05, 4e05, 5e05]

# MIA and DCR vs. Diffusion steps
diffusion_steps_dcr = [0.71149, 0.73510, 0.76400, 0.77149, 0.70600, 0.71430, 0.71000, 0.74450, 0.70990, 0.70969]
diffusion_steps_wb = [0.18900, 0.24300, 0.25600, 0.26700, 0.44200, 0.45500, 0.44000, 0.44500, 0.43700, 0.42600]
diffusion_steps_bb = [0.13000, 0.18600, 0.20100, 0.22700, 0.26000, 0.26100, 0.27100, 0.25200, 0.25900, 0.25500]
diffusion_steps = [10, 20, 50, 80, 100, 500, 1000, 2000, 3000, 4000]

# MIA and DCR vs. Synthetic data
synthetic_size = ["1x", "10x", "1M"]
bb_10k = [0.34, 0.48, 0.48]
bb_20k = [0.24, 0.33, 0.35]
bb_50k = [0.11, 0.14, 0.14]
bb_100k = [0.1, 0.112, 0.112]
dcr_20k = [0.749, 0.748, 0.749]

# MIA and DCR vs. Batch Size
batch_size_dcr = [0.7490, 0.7445, 0.7922]
batch_size_wb = [0.4390, 0.4450, 0.484]
batch_size_bb = [0.2450, 0.2520, 0.2550]
batch_size = [2048, 4096, 8192]

train_size = [5e03, 1e04, 2e04, 5e04, 1e05]
train_size_delta_dcr = [0.28900, 0.18000, 0.08800, 0.00800, 0.02000]
train_size_wb = [0.60700, 0.57000, 0.43000, 0.19100, 0.13900]
train_size_bb = [0.40000, 0.34000, 0.24000, 0.11000, 0.10000]

shadow_models = [1, 5, 10, 20, 50]
shadow_models_wb = [0.423, 0.434, 0.440, 0.458, 0.459]
shadow_models_bb = [0.241, 0.287, 0.267, 0.275, 0.271]
shadow_models_bb_roc = [0.626, 0.655, 0.644, 0.645, 0.652]
shadow_models_wb_roc = [0.775, 0.786, 0.788, 0.792, 0.798]

This notebook will visualize how each of the following variable affects the fairness towards a particular protected group.

In [3]:
FIG_HEIGHT = 800
FIG_WIDTH = 800
FONT_SIZE = 32

## Utilities

### Plotting Utils

In [4]:
def format_metric_name(metric_name: str) -> str:
    """
    Prettify diagram texts by replacing
    snake case with regular title case.
    """
    output_words = []
    for word in metric_name.split("_"):
        word = word.title() if word not in ["FPR", "TPR"] else word.upper()
        output_words.append(word)

    return " ".join(output_words)

## Plotting

In [5]:
repo_abs_path = Path(os.path.abspath("")).parent.parent

plots_dir = f"{repo_abs_path}/examples/visualizations/"
plot_name = "mia_dcr_results"

In [6]:
def customize_figure_layout(gen_fig: Any) -> None:
    gen_fig.update_layout(
        height=FIG_HEIGHT,
        width=FIG_WIDTH,
        font_color="black",
        legend=dict(
            y=-0.3,
            x=0.5,
            xanchor="center",
            yanchor="bottom",
            orientation="h",
            valign="top",
            title_text="",
            font=dict(size=FONT_SIZE-4),
            title_font_family="Helvetica",
        ),
        title_font_family="Helvetica",
        title_x=0.5,
        title_y=0.975,
        margin=dict(l=30, r=30, t=75, b=25),
        plot_bgcolor="#eeeeee",
        font=dict(size=FONT_SIZE, family="Helvetica"),
    )


def save_and_display_figure(gen_fig: Any, template_name: str, plots_dir: str) -> None:
    plot_png_path = f"{plots_dir}/{template_name}.png"
    plot_pdf_path = f"{plots_dir}/{template_name}.pdf"

    os.makedirs(plots_dir, exist_ok=True)
    gen_fig.write_image(plot_png_path, scale=2)
    gen_fig.write_image(plot_pdf_path)

    display(gen_fig)

In [7]:
# Train steps figure
df = {"Train Steps": train_steps, "MIA (WB)": train_steps_wb, "MIA (BB)": train_steps_bb, "DCR": train_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Train Steps",
    y=["MIA (WB)", "MIA (BB)", "DCR"],
    markers=True,
    title="MIA and DCR vs. Training Steps",
)
fig.update_layout(
    xaxis=dict(
        tickmode='array',  # Set tickmode to 'array' to use tickvals/ticktext
        tickvals=[5e03, 1e04, 2e04, 5e04, 1e05, 2e05, 5e05],   # Specify the x-coordinates of your data points as tick values
    ),
)
fig.update_xaxes(
    type="log", matches=None, tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    matches=None, title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "mia_dcr_train_steps", plots_dir)

In [8]:
# Diffusion steps figure
df = {"Diffusion Steps": diffusion_steps, "MIA (WB)": diffusion_steps_wb, "MIA (BB)": diffusion_steps_bb, "DCR": diffusion_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Diffusion Steps",
    y=["MIA (WB)", "MIA (BB)", "DCR"],
    markers=True,
    title="MIA and DCR vs. Diffusion Steps",
)
fig.update_layout(
    xaxis=dict(
        tickmode='array',  # Set tickmode to 'array' to use tickvals/ticktext
        tickvals=[1e01, 2e01, 5e01, 1e02, 2e02, 5e02, 1e03, 2e03, 5e03],   # Specify the x-coordinates of your data points as tick values
    ),
)
fig.update_xaxes(
    type="log", matches=None, tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    matches=None, title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "mia_dcr_diffusion_steps", plots_dir)

In [9]:
# Synthetic Size figure
df = {"Synthetic Size": synthetic_size, "MIA (BB) 10K": bb_10k, "MIA (BB) 20K": bb_20k, "MIA (BB) 50K": bb_50k, "MIA (BB) 100K": bb_100k, "DCR 20K": dcr_20k}

fig = px.line(
    data_frame=df,
    x="Synthetic Size",
    y=["MIA (BB) 10K", "MIA (BB) 20K", "MIA (BB) 50K", "MIA (BB) 100K", "DCR 20K"],
    markers=True,
    title="MIA and DCR vs. Synthetic Size",
)
fig.update_layout(
    xaxis=dict(
        tickmode='array',  # Set tickmode to 'array' to use tickvals/ticktext
        tickvals=["1x", "10x", "1M"],   # Specify the x-coordinates of your data points as tick values
    ),
)
fig.update_xaxes(
    matches=None, tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    matches=None, title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "mia_dcr_synthetic_size", plots_dir)

In [10]:
# Batch size figure
df = {"Batch Size": batch_size, "MIA (WB)": batch_size_wb, "MIA (BB)": batch_size_bb, "DCR": batch_size_dcr}

fig = px.line(
    data_frame=df,
    x="Batch Size",
    y=["MIA (WB)", "MIA (BB)", "DCR"],
    markers=True,
    title="MIA and DCR vs. Batch Size",
)
fig.update_layout(
    xaxis=dict(
        tickmode='array',  # Set tickmode to 'array' to use tickvals/ticktext
        tickvals=["2048", "4096", "8192"],   # Specify the x-coordinates of your data points as tick values
    ),
)
fig.update_xaxes(
    matches=None, tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    matches=None, title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "mia_dcr_batch_size", plots_dir)

In [11]:
# Training size figure
df = {"Train Size": train_size, "MIA (WB)": train_size_wb, "MIA (BB)": train_size_bb, "Delta DCR": train_size_delta_dcr}

fig = px.line(
    data_frame=df,
    x="Train Size",
    y=["MIA (WB)", "MIA (BB)", "Delta DCR"],
    markers=True,
    title="MIA and DCR vs. Train Size",
)
fig.update_layout(
    xaxis=dict(
        tickmode='array',  # Set tickmode to 'array' to use tickvals/ticktext
        tickvals=train_size,   # Specify the x-coordinates of your data points as tick values
    ),
)
fig.update_xaxes(
    type="log", matches=None, tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E",
)
fig.update_yaxes(
    matches=None, title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "mia_dcr_train_size", plots_dir)

In [12]:
# Batch size figure
df = {"Shadow Models": shadow_models, "MIA (WB)": shadow_models_wb, "MIA (BB)": shadow_models_bb, "ROC/AUC (WB)": shadow_models_wb_roc, "ROC/AUC (BB)": shadow_models_bb_roc}

fig = px.line(
    data_frame=df,
    x="Shadow Models",
    y=["MIA (WB)", "ROC/AUC (WB)",  "MIA (BB)", "ROC/AUC (BB)"],
    markers=True,
    title="MIA and ROC/AUC vs. Shadow Models",
)
fig.update_layout(
    xaxis=dict(
        tickmode='array',  # Set tickmode to 'array' to use tickvals/ticktext
        tickvals=shadow_models,   # Specify the x-coordinates of your data points as tick values
    ),
)
fig.update_xaxes(
    type="log", matches=None, tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    matches=None, title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
save_and_display_figure(fig, "mia_dcr_shadow_models", plots_dir)